# Nepal Disaster Impact: HDBSCAN, GMM, and Linear Regression

This notebook runs the reproducible pipeline against the supplied BIPAD incident/loss dataset and inspects the generated outputs. It is an implementation notebook, not the research report.

**Research questions**
1. Do HDBSCAN and Gaussian Mixture Models (GMM) identify interpretable incident-impact profiles?
2. How much does HDBSCAN classify as noise, and how do internal clustering metrics compare?
3. How accurately can Linear Regression estimate reported `estimatedLoss` from reported incident-impact counts and context features?

**Important:** loss estimates are retrospective estimates based on post-incident records, not pre-disaster forecasts. The currency/unit of the monetary fields has not been independently verified.

In [ ]:
from pathlib import Path
import sys
import subprocess
import pandas as pd
import numpy as np
import plotly.express as px

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
SCRIPT = PROJECT_ROOT / "src" / "run_clustering_experiments.py"
REPORTS = PROJECT_ROOT / "reports"
DATA = PROJECT_ROOT / "data" / "raw" / "Nepal_Disaster_Data_2020_2026.csv"
print("Project root:", PROJECT_ROOT)
print("Dataset exists:", DATA.exists())
print("Pipeline exists:", SCRIPT.exists())

## 1. Run the complete pipeline

The script fits HDBSCAN configurations and GMM component counts on the same reproducible sample, evaluates clustering metrics, and trains/tests a log-target Linear Regression model. The data split and random seed are fixed for reproducibility.

In [ ]:
result = subprocess.run([sys.executable, str(SCRIPT)], cwd=PROJECT_ROOT, text=True, capture_output=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Pipeline failed; review the error above.")

## 2. Compare HDBSCAN and GMM

Silhouette is generally higher-is-better; Davies–Bouldin is lower-is-better; Calinski–Harabasz is higher-is-better. For HDBSCAN, these three metrics are computed after excluding noise labels, and `noise_fraction` is reported separately. A metric should not be used in isolation to declare a winner.

In [ ]:
comparison = pd.read_csv(REPORTS / "clustering_model_comparison.csv")
comparison.sort_values(["algorithm", "silhouette"], ascending=[True, False], na_position="last")

In [ ]:
valid_metrics = comparison.dropna(subset=["silhouette"])
if not valid_metrics.empty:
    display(valid_metrics.sort_values("silhouette", ascending=False).head(10))
    fig = px.bar(valid_metrics, x="algorithm", y="silhouette", color="algorithm",
                 hover_data=[c for c in ["min_cluster_size", "min_samples", "n_components", "noise_fraction", "clusters_found"] if c in valid_metrics],
                 title="Silhouette score by tested clustering configuration")
    fig.show()
else:
    print("No configuration produced a valid silhouette score; inspect cluster counts/noise.")

## 3. Inspect cluster profiles

Cluster numbers are arbitrary IDs, not low/medium/high severity classes. Review medians and the most common hazard before assigning meaning. HDBSCAN `-1` labels indicate noise or observations outside the fitted sample.

In [ ]:
assignments = pd.read_csv(REPORTS / "impact_cluster_assignments.csv")
profiles = pd.read_csv(REPORTS / "impact_cluster_profiles.csv")
display(profiles.head(30))
print("Rows with HDBSCAN labels:", assignments["hdbscan_cluster"].notna().sum())
print("HDBSCAN noise/unassigned fraction:", (assignments["hdbscan_cluster"] == -1).mean())
print("GMM cluster sizes:")
display(assignments["gmm_cluster"].value_counts().sort_index().rename_axis("GMM cluster").to_frame("incident_records"))

## 4. Linear Regression for loss estimation

The target is `estimatedLoss`. Rows with missing target values are excluded from fitting and testing, not treated as zero. The model fits `log1p(loss)` and converts predictions back using `expm1`. Loss component columns are excluded from predictors to reduce leakage. Since the source target is highly skewed, original-scale errors can be dominated by extreme incidents.

In [ ]:
loss_metrics = pd.read_csv(REPORTS / "loss_regression_metrics.csv")
display(loss_metrics.T.rename(columns={0: "value"}))

In [ ]:
loss_predictions = pd.read_csv(REPORTS / "estimated_loss_predictions.csv")
print("Total incident rows:", len(loss_predictions))
print("Rows with reported loss:", int(loss_predictions["loss_was_reported"].sum()))
print("Rows with missing reported loss:", int((~loss_predictions["loss_was_reported"]).sum()))
missing_loss_estimates = loss_predictions.loc[~loss_predictions["loss_was_reported"]].copy()
display(missing_loss_estimates.sort_values("linear_regression_predicted_loss", ascending=False).head(20))

## 5. Interpretation and limitations

- HDBSCAN is fitted on a sample and does not assign new rows outside that sample in this implementation; those rows receive `-1` alongside noise observations.
- GMM is fitted on the same sample and predicts components for all rows.
- Internal clustering scores can be misleading for zero-inflated impact data; inspect noise fraction, cluster sizes, and actual median profiles.
- The loss model is a baseline. A weak original-scale R² means the current feature set does not explain much of the variation in monetary loss.
- The regression inputs are post-event reported impacts; the model should not be described as an early-warning forecast.
- Verify the currency/unit and source semantics for zero values before using estimated totals. Missing loss and zero loss are not equivalent.
- Results describe the supplied extract and its reporting practices, not necessarily all disasters in Nepal.